# PixelCNN — autoregressive images with masked convolutions

> Tutorial pair for [`pixelcnn.py`](pixelcnn.py).

## 1. Intuition
Read an image like text: left-to-right, top-to-bottom. PixelCNN predicts each
pixel from only the pixels that came before it in this raster scan. The clever
part is doing this with ordinary convolutions: by **masking** the kernel so it
can never look at the current or future pixels, one convolution computes every
pixel's conditional distribution at once during training -- yet generation still
proceeds strictly one pixel at a time.

## 2. Concept (the slide)
- Factor the image likelihood as a product of per-pixel conditionals.
- Enforce that ordering with **masked convolutions**.
- **Mask A** (first layer): a pixel may not see its own value -> no cheating.
- **Mask B** (deeper layers): a pixel may see its own *feature* (which already
  only summarizes the past), so information still never flows backward.
- Train by maximum likelihood (teacher forcing); sample pixel by pixel.

## 3. Math derivation — factorization & masking

**Autoregressive factorization.** Any joint distribution factorizes exactly via
the chain rule. Order the $N=H\cdot W$ pixels in raster scan $x_1,\dots,x_N$:
$$\boxed{\,p(x)=\prod_{i=1}^{N}p\big(x_i\mid x_1,\dots,x_{i-1}\big)\,}.$$
No approximation -- this is just conditional probability. The model only needs to
parameterize each conditional. For binary pixels we use a Bernoulli with logit
$\ell_i=f_\theta(x_{<i})$:
$$p(x_i\mid x_{<i})=\sigma(\ell_i)^{x_i}\,(1-\sigma(\ell_i))^{1-x_i}.$$

**Log-likelihood / loss.** Negative log-likelihood is a sum of per-pixel binary
cross-entropies, which is *exact* (unlike the VAE's bound):
$$-\log p(x)=\sum_{i=1}^{N}\big[-x_i\log\sigma(\ell_i)-(1-x_i)\log(1-\sigma(\ell_i))\big].$$

**Why masking enforces the ordering.** A conv output at pixel $i$ must depend only
on $x_{<i}$. Zero the kernel weights covering the current and future positions:
for a $k\times k$ kernel with center $c=\lfloor k/2\rfloor$,
$$M[r,:]=0\ \text{for } r>c,\qquad M[c,\,c{+}1{:}]=0,$$
and for the **type-A** mask additionally $M[c,c]=0$. The first conv uses mask A
(it touches raw pixel values, so it must exclude $x_i$ itself); all deeper convs
use **type B**, where the center channel is a *feature* of $x_{<i}$, so allowing
$M[c,c]=1$ is safe and gives the network access to its own receptive field.
Because every layer respects causality, the composition does too: a single forward
pass yields all $\ell_i$ with **no leakage** from later pixels (teacher forcing).

**Generation.** Sampling cannot be parallel: draw $x_1\sim p(x_1)$, feed it back,
draw $x_2\sim p(x_2\mid x_1)$, and so on for all $N$ pixels.

## 4. Model — masked convolution + the PixelCNN stack

In [ ]:
# ===== actual implementation from pixelcnn.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _make_mask_numpy(k: int, mask_type: str) -> np.ndarray:
    m = np.ones((k, k), dtype=np.float32)
    c = k // 2
    m[c, c + 1:] = 0.0      # center row, future columns
    m[c + 1:, :] = 0.0      # all rows below
    if mask_type == "A":
        m[c, c] = 0.0       # type A also blocks the current pixel
    return m

import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class MaskedConv2d(nn.Conv2d):
    r"""
    A Conv2d whose kernel is multiplied by a causal mask before every forward
    pass. Type A excludes the center pixel (used once, on the input); type B
    includes it (used in all deeper layers, where the center channel is already a
    *feature* of past pixels, not the pixel value itself).
    """

    def __init__(self, mask_type: str, *args, **kwargs):
        super().__init__(*args, **kwargs)
        assert mask_type in ("A", "B")
        k = self.kernel_size[0]
        mask = torch.ones_like(self.weight)            # (out, in, k, k)
        c = k // 2
        mask[:, :, c, c + 1:] = 0.0
        mask[:, :, c + 1:, :] = 0.0
        if mask_type == "A":
            mask[:, :, c, c] = 0.0
        self.register_buffer("mask", mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.weight.data *= self.mask                  # zero the "future" weights
        return super().forward(x)

class PixelCNN(nn.Module):
    """Tiny PixelCNN for binary 1x8x8 images (Bernoulli per pixel)."""

    def __init__(self, channels: int = 32, n_layers: int = 5, k: int = 5):
        super().__init__()
        layers = [MaskedConv2d("A", 1, channels, k, padding=k // 2), nn.ReLU()]
        for _ in range(n_layers):
            layers += [MaskedConv2d("B", channels, channels, k, padding=k // 2),
                       nn.ReLU()]
        layers += [nn.Conv2d(channels, 1, 1)]          # 1x1: per-pixel logit
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return per-pixel Bernoulli logits, shape like x."""
        return self.net(x)

    def loss(self, x: torch.Tensor) -> torch.Tensor:
        """Negative log-likelihood (bits are summed over pixels)."""
        logits = self(x)
        return F.binary_cross_entropy_with_logits(logits, x, reduction="none") \
            .sum(dim=[1, 2, 3]).mean()

    def fit(self, X, epochs: int = 30, batch: int = 128, lr: float = 1e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                loss = self.loss(X[perm[s:s + batch]])
                opt.zero_grad(); loss.backward(); opt.step()
                tot += loss.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def sample(self, n: int, size: int = 8):
        """Ancestral sampling in raster order: fill one pixel at a time."""
        dev = next(self.parameters()).device
        x = torch.zeros(n, 1, size, size, device=dev)
        for i in range(size):
            for j in range(size):
                logits = self(x)
                p = torch.sigmoid(logits[:, :, i, j])
                x[:, :, i, j] = torch.bernoulli(p)     # only this pixel is committed
        return x.cpu().numpy()

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1)  # tiny model: 1 thread avoids CPU thrashing
    from sklearn.datasets import load_digits
    X = load_digits().data.reshape(-1, 1, 8, 8) / 16.0
    X = (X > 0.3).astype(np.float32)                   # binarize to {0,1}

    m = PixelCNN(channels=32, n_layers=4).fit(X, epochs=25)
    nll = m.history[-1]
    print(f"PixelCNN final NLL = {nll:.2f} nats/image "
          f"({nll / 64:.3f} nats/pixel)")

    s = m.sample(16)
    print(f"  sampled {s.shape[0]} images, mean on-pixels="
          f"{s.mean():.3f} (data {X.mean():.3f})")

## 5. Training / sampling — exact NLL loss + raster-order sampler

In [ ]:
# ===== actual implementation from pixelcnn.py =====

## 6. Train & sample on binarized 8×8 digits

In [ ]:
demo()

## 7. Visualization — the type A/B masks and generated digits

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
import pixelcnn as M

# the causal masks themselves
maskA = M._make_mask_numpy(5, "A")
maskB = M._make_mask_numpy(5, "B")
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
for a, (m, t) in zip(ax, [(maskA, "mask A"), (maskB, "mask B")]):
    a.imshow(m, cmap="gray", vmin=0, vmax=1); a.set_title(t)
    a.set_xticks(range(5)); a.set_yticks(range(5))
plt.tight_layout(); plt.show()

# train and sample
X = load_digits().data.reshape(-1, 1, 8, 8) / 16.0
X = (X > 0.3).astype("float32")
m = M.PixelCNN(channels=32, n_layers=4).fit(X, epochs=25)
samples = m.sample(16)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, a in enumerate(axes.ravel()):
    a.imshow(samples[i, 0], cmap="gray"); a.axis("off")
fig.suptitle("PixelCNN samples (generated pixel by pixel)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- The autoregressive factorization is *exact*: PixelCNN gives a true (tractable)
  log-likelihood, unlike VAEs (a bound) or GANs (none).
- The **mask** is the whole idea -- get mask A vs B wrong and the model either
  cheats (sees the answer) or loses its own receptive field.
- **Blind spot**: a naive masked conv cannot see some pixels above-right; the
  Gated PixelCNN fixes this with separate horizontal/vertical stacks.
- Sampling is inherently sequential and slow ($O(N)$ forward passes); training is
  fully parallel.
- This discrete autoregressive model is exactly what is used as the **prior over
  VQ-VAE codes** to turn that discrete encoder into a generator.